<a href="https://colab.research.google.com/github/ZhangPeiLin2004/Duty-of-Care/blob/main/lab01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laptop lab 1: Visualization
*Look at your data before you model it*

**Machine Learning with Python** · Minor Amsterdam Data Science and Artificial Intelligence · Fall 2026<br>
Thursday, October 29, 2026 · Laptop labs: Batuhan Ataş (b.atas@uva.nl)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batuatas/MLwPython-labs/blob/main/lab01.ipynb)

> **Before you start:** choose **File → Save a copy in Drive** and work in that copy. Colab does not save your changes in the original notebook.

In Lecture 1 you saw what machine learning is, and the workflow of a machine learning project: it starts with the problem and the data, long before any model. In Lecture 2 you saw how to plot data, and what plots reveal about the quality of data. Today you put both into practice, on real data: the Airbnb listings of Amsterdam.

**The story.** A friend of yours wants to rent out her apartment in Amsterdam on Airbnb, and asks you: *what can I ask per night?* You have a table with more than 10,000 Amsterdam listings, collected from the Airbnb website on September 16, 2026. In the words of Lecture 1, her question is a supervised learning problem: each listing is an example, its price is the label, and its size, location and other properties are the features. Before anyone, or any model, can answer her question, the data must be checked and cleaned. That is what this lab is about:

1. **A first look** (Task 1): what is in the table, and what is wrong with it?
2. **Cleaning, step by step** (Task 2): we repair the problems one by one, and we explain every choice.
3. **What goes with the price?** (Task 3): which features go together with the label?
4. **Reviews over time** (Task 4): a line plot, and two series on twin axes.
5. **Back to your friend** (Task 5): a first answer to her question.

## The data

[Inside Airbnb](https://insideairbnb.com) collects the public listings of Airbnb in many cities. The lab downloads two of their files of Amsterdam directly from their website: the listings, one row per listing, and the reviews, one row per review (only the listing and the date). The listings file has 90 columns; we keep the 20 below.

| Column | Meaning |
|---|---|
| `neighbourhood` | one of the 22 neighborhoods of Amsterdam (the data use British spelling) |
| `room_type` | `Entire home/apt`, `Private room`, `Hotel room` or `Shared room` |
| `property_type` | as the host describes it, for example `Entire rental unit` or `Boat` |
| `accommodates`, `bedrooms`, `beds` | maximum number of guests, number of bedrooms and beds |
| `price` | price per night **in euros**; the `$` sign is a technical artifact of the export |
| `minimum_nights`, `maximum_nights` | shortest and longest stay the host allows |
| `availability_365` | number of nights that can still be booked in the coming 365 days |
| `number_of_reviews`, `review_scores_rating` | number of reviews, and the average rating (at most 5) |
| `first_review` | date of the first review |
| `host_is_superhost` | `t` (true) or `f` (false): Airbnb's label for experienced, highly rated hosts |
| `host_listings` | number of listings this host has in Amsterdam |
| `latitude`, `longitude` | location of the listing |

## How this lab works

Most of the code is given: read it, run it, and look carefully at the output. What matters is what you see, and what you conclude from it. In the plots, you sometimes fill in a part yourself, such as the labels or the colors.

| You see | What you do |
|---|---|
| **Run** | Run the cell and look at the output. |
| **Your turn** | Fill in the dots `...` in a plot: a label, a color, a number. Until you do, the cell shows an orange *Not yet* box: that is normal. |
| **Check** | Run it right after your own cell. It says ✓ when your code works, or tells you what to look at. |
| **Hint** | Click to unfold it when you are stuck. |
| **Think** | Double-click *Your answer* and write a sentence or two. We discuss the answers together. |

Run the cells from top to bottom with **Shift + Enter**: later cells use the results of earlier ones. Stuck for more than five minutes? Raise your hand: that is what the lab is for.

| Task | Minutes |
|---|---|
| 0. Setup | 5 |
| 1. A first look | 12 |
| 2. Cleaning, step by step | 20 |
| 3. What goes with the price? | 36 |
| 4. Reviews over time | 12 |
| 5. Back to your friend | 5 |
| Check your understanding | together, at the end |
| If you finish early | at home |

## Task 0. Setup

Run the cell below. It loads the Python libraries you need today, downloads the checks of this lab and the data from Inside Airbnb, and tells you whether everything works. The download takes a few seconds. You should see a green box with a ✓.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# The checks of this lab are in the course lab repository;
# setup_ok() also downloads the data from Inside Airbnb.
BASE = "https://raw.githubusercontent.com/batuatas/MLwPython-labs/main/"
import urllib.request
urllib.request.urlretrieve(BASE + "lab01_tools.py", "lab01_tools.py")
from lab01_tools import *

setup_ok()

## Task 1. A first look

In this task you meet the table, and you look for problems before you trust it.

**Run.** Load the listings and show the first five rows. In pandas, a table is called a *DataFrame*: one row per listing, one column per variable.

In [4]:
listings = load_listings()     # the listings of Inside Airbnb, with the 20 columns we use
listings.head()

,id,neighbourhood,room_type,property_type,accommodates,bedrooms,beds,price,minimum_nights,maximum_nights,availability_365,number_of_reviews,first_review,last_review,review_scores_rating,host_is_superhost,host_listings,license,latitude,longitude
0,28871,Centrum-West,Private room,Private room in rental unit,2,1.0,1.0,$97.00,2.0,730.0,49,815,2010-08-22,2026-08-29,4.86,t,2,0363 607B EA74 0BD8 2F6F,52.36775,4.89092
1,29051,Centrum-Oost,Private room,Private room in condo,2,1.0,1.0,NaN,2.0,730.0,0,929,2011-03-16,2026-08-29,4.82,t,2,0363 607B EA74 0BD8 2F6F,52.36584,4.89111
2,44391,Centrum-Oost,Entire home/apt,Entire rental unit,4,2.0,2.0,NaN,3.0,730.0,0,42,2010-09-16,2022-08-20,4.71,f,1,0363 E76E F06A C1DD 172C,52.37168,4.91471
3,48373,Buitenveldert - Zuidas,Entire home/apt,Entire rental unit,4,2.0,2.0,NaN,3.0,1125.0,0,5,2023-07-21,2024-04-28,5.00,f,1,0363 4A2B A6AD 0196 F684,52.32781,4.87680
4,49552,Centrum-West,Entire home/apt,Entire guest suite,3,2.0,2.0,$424.29,1.0,1125.0,270,667,2010-10-29,2026-09-07,4.94,t,1,0363 576A D827 5085 6B83,52.38028,4.89089


**Think.** Lecture 1, "Supervised learning": one row per example, one column per feature, one column with the label. For your friend's question, which column is the label? Name three columns you would use as features. Is it a classification or a regression problem? And name a column that you could predict as a *classification* problem instead.

*Your answer:*

**Your turn.** Lecture 2 gave four lines to run on every new dataset, before any plot ("Four lines before any plot"). Write them yourself for `listings`, one per cell, so that you can read each output on its own:

- `dtypes`: the type of each column. Numbers show as `int64` or `float64`, text as `object` or `str`.
- `isna().sum()`: the number of missing values in each column.
- `duplicated().sum()`: the number of rows that appear twice.
- `describe()`: summary statistics of the numeric columns: count, mean, smallest value, largest value, and so on. Large numbers are written in scientific notation: `2.147484e+09` means 2.147484 × 10⁹, about 2 billion.

In [5]:
# the type of each column
listings.dtypes

,0
id,int64
neighbourhood,object
room_type,object
property_type,object
accommodates,int64
bedrooms,float64
beds,float64
price,object
minimum_nights,float64
maximum_nights,float64


In [6]:
# missing values per column
listings.isna().sum()

,0
id,0
neighbourhood,0
room_type,0
property_type,0
accommodates,0
bedrooms,29
beds,385
price,4573
minimum_nights,268
maximum_nights,268


In [7]:
# number of duplicate rows
listings.duplicated().sum()

np.int64(0)

In [8]:
# summary statistics
listings.describe()

,id,accommodates,bedrooms,beds,minimum_nights,maximum_nights,availability_365,number_of_reviews,review_scores_rating,host_listings,latitude,longitude
count,1.061200e+04,10612.000000,10583.000000,10227.000000,10344.000000,1.034400e+04,10612.000000,10612.000000,9610.000000,10612.000000,10612.000000,10612.000000
mean,7.314719e+17,2.929325,1.531796,1.830644,3.792827,4.155551e+05,93.621466,53.894365,4.848943,1.830758,52.366829,4.889039
std,6.409643e+17,1.266601,0.880448,1.358225,15.747927,2.985929e+07,123.624953,147.699524,0.247408,2.693007,0.017163,0.034760
min,2.887100e+04,1.000000,0.000000,1.000000,1.000000,1.000000e+00,0.000000,0.000000,1.000000,1.000000,52.290280,4.755870
25%,3.050003e+07,2.000000,1.000000,1.000000,2.000000,2.000000e+01,0.000000,3.000000,4.800000,1.000000,52.355859,4.864307
50%,8.339275e+17,2.000000,1.000000,1.000000,2.000000,3.100000e+01,14.000000,11.000000,4.920000,1.000000,52.365819,4.887360
75%,1.303882e+18,4.000000,2.000000,2.000000,3.000000,3.650000e+02,178.250000,35.000000,5.000000,1.000000,52.376540,4.908143
max,1.774618e+18,16.000000,17.000000,33.000000,800.000000,2.147484e+09,365.000000,5732.000000,5.000000,24.000000,52.425160,5.028150


<details><summary><b>Hint</b> (click to unfold)</summary>

The slide of Lecture 2, where the table was called `raw`:

```python
raw.dtypes              # types per column
raw.isna().sum()        # missing values
raw.duplicated().sum()  # duplicate rows
raw.describe()          # min, max, mean, ...
```

Here the table is called `listings`.

</details>

**Think.** Look at the column `price` in these outputs, and compare it with the codebook. Which problems do you see? (Also look at the largest value of `maximum_nights` in `describe()`.)

*Your answer:* 1. Incorrect data type and missing values in price, 2. An implausible maximum value for maximum_nights

## Task 2. Cleaning, step by step

Cleaning means repairing the problems you found, *before* you plot or model. There is rarely one correct way to do it: each step is a choice, and a good data scientist can explain every choice. You do not need to write the cleaning code yourself today: it is given, with comments. What matters is that you understand *what* is wrong and *why* we repair it the way we do. We take four steps.

### Step 1. Text to numbers

The prices are text such as `"$97.00"`, and prices of 1,000 euros or more also contain a comma, such as `"1,175.00"`. Python cannot compute with text, so we remove the `$` and the `,` and convert what is left to a number.

**Run.**

In [ ]:
listings["price"] = (listings["price"]
                     .str.replace("$", "")      # "$1,175.00" -> "1,175.00"
                     .str.replace(",", "")      # "1,175.00"  -> "1175.00"
                     .astype(float))            # "1175.00"   -> 1175.0
listings["price"].describe()

### Step 2. Missing values

More than 40% of the prices are missing. There are three common ways to deal with missing values:

1. **drop** the rows without a value;
2. **fill** each gap with one value, for example the median (this is called *imputation*);
3. **keep** the gaps, and only use the rows that have a value where needed.

Which one is right depends on *why* the values are missing. So first we look at who is missing.

**Run.** The two histograms show, for the listings with and without a price, how many nights can still be booked in the coming year. The bar chart shows which share of each room type has no price.

In [ ]:
no_price = listings["price"].isna()     # True for every listing without a price

# subplots(1, 3): 1 row of 3 plot areas, axes[0] to axes[2]
# figsize=(width, height) in inches; layout="constrained" keeps the labels from overlapping
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), layout="constrained")
# bins=37: 37 bars; range=(0, 370): from 0 to 370 nights, so 10 nights per bar
axes[0].hist(listings.loc[~no_price, "availability_365"], bins=37, range=(0, 370))
axes[0].set_title("listings with a price")
axes[1].hist(listings.loc[no_price, "availability_365"], bins=37, range=(0, 370),
             color="tab:orange")   # a color name, in quotes
axes[1].set_title("listings without a price")
for ax in axes[:2]:
    ax.set_xlabel("nights still bookable in the coming year")
    ax.set_ylabel("listings")

share = no_price.groupby(listings["room_type"]).mean().sort_values()   # share without a price
axes[2].barh(share.index, share.values)   # horizontal bars: names, then lengths
axes[2].set_xlabel("share of listings without a price")
plt.show()

**Think.** What do the plots tell you about the listings without a price? Which of the three options would you choose, and why? Write down your answer before you read on.

*Your answer:*

Almost every listing without a price cannot be booked on any night of the coming year, and half of the whole homes have no price. The reason: the website only shows a price for nights that can be booked, and in Amsterdam a whole home may be rented out to tourists for at most 30 nights a year. Many homes have used up their nights, or are rented out only a few weekends a year, so their calendar is closed.

So the prices are not missing at random: whether a price is missing depends on the kind of listing. Lecture 2 asked: who did not answer, and why? This is the answer.

**Run.** What would *filling* do? Here every missing price is filled with the median price.

In [ ]:
filled = listings["price"].fillna(listings["price"].median())   # fill the gaps with the median

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)   # sharey: one y-axis for both
axes[0].hist(listings["price"], bins=60, range=(0, 1000))
axes[0].set_title("only the listings with a price")
axes[1].hist(filled, bins=60, range=(0, 1000))
axes[1].set_title("missing prices filled with the median")
for ax in axes:
    ax.set_xlabel("price per night (euros)")
axes[0].set_ylabel("listings")
plt.show()

Filling creates one tall spike: more than 4,000 listings with exactly the same price, a pile-up at one value as in "Suspiciously round numbers" (Lecture 2). Worse, it gives the closed homes the price of the open ones: we would invent prices that nobody asked.

**Our choice:** we *drop* the listings without a price. A price we do not know cannot help our friend. But we keep in mind what we lose: mostly whole homes with a closed calendar, so our answer is about homes that are rented out regularly. (Lecture 7 treats missing values in depth.)

In [ ]:
priced = listings[listings["price"].notna()]   # keep only the rows that have a price
len(priced)

### Step 3. Values that are codes, not measurements

`describe()` showed a `maximum_nights` of 2,147,483,647. That is $2^{31} - 1$, the largest number that fits in a standard 32-bit integer: a computer system's way of saying "no limit". It is not a number of nights, just like 9999 in many other datasets, or the default location in Kansas of Lecture 2. Left as it is, it ruins every mean and every plot of the column, so we replace it with `np.nan`, the value pandas uses for *missing*.

**Run.** Look at the most common values first: 1,125 nights is also suspicious. It is Airbnb's default setting, so it tells you that the host never changed the setting, not what the host wants. We leave it, but we keep it in mind.

In [ ]:
print(priced["maximum_nights"].value_counts().head())
priced = priced.replace({"maximum_nights": {2147483647: np.nan}})   # a code, not a number of nights
print("largest value now:", priced["maximum_nights"].max())

### Step 4. Outliers

An *outlier* is a value far away from the others. Some outliers are errors, some are real and rare. Before we decide what to do with them, we must find and look at them.

To compare the prices of the room types, we draw *box plots* side by side. Each box holds the middle half of the prices of one room type, and the line inside is the median. The whiskers reach to the usual prices, and every price beyond them is drawn as a separate dot.

**Run.**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=priced, x="room_type", y="price", ax=ax)   # data: the table; x, y: column names
ax.set_xlabel("room type")
ax.set_ylabel("price per night (euros)")
ax.set_title("Before removing outliers")
plt.show()

The boxes are squashed into a thin strip at the bottom, because a few dots lie far above all the others: the y-axis has to run to almost €12,000 to fit them. These are the outliers.

**Run.** The six most expensive listings, with the columns that help to judge them.

In [ ]:
cols = ["price", "property_type", "accommodates", "bedrooms", "neighbourhood", "number_of_reviews"]
priced.sort_values("price", ascending=False)[cols].head(6)

**Think.** Which of these prices do you believe, and which not? Give a possible reason for one that you doubt. Then decide: would you remove any of these listings before you answer your friend?

*Your answer:*

**Our choice.** Our friend has an ordinary apartment. Party boats and prices that nobody seems to pay do not help her, so we leave out the listings of €2,000 a night or more: 10 of the 6,039. Note that this is a choice *for this question*: a study of party boats would keep exactly these rows. And outliers are not always noise: in fraud detection, for example, the rare cases are the whole point.

In [ ]:
clean = priced[priced["price"] < 2000]
print(len(priced) - len(clean), "listings left out;", len(clean), "listings in the clean table")

**Run.** The same box plots, after removing them. Now you can read the boxes: compare the medians and the spread of the room types.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=clean, x="room_type", y="price", ax=ax)
ax.set_xlabel("room type")
ax.set_ylabel("price per night (euros)")
ax.set_title("After removing outliers")
plt.show()

There are still dots above the whiskers: prices that are high for their room type, but believable, such as large homes. We keep them. Removing every dot would throw away real information.

**Summary of the cleaning.** Every step was a choice, and here is why:

| Problem | What we did | Why |
|---|---|---|
| prices stored as text | removed `$` and `,`, converted to numbers | we cannot compute with text |
| 43% of the prices missing | dropped those listings | an unknown price cannot help; filling would invent prices |
| `maximum_nights` of 2,147,483,647 | replaced by `np.nan` | a code for "no limit", not a number of nights |
| a few prices of €2,000 or more | left out for this question | not comparable with an ordinary apartment |

From now on, we work with `clean`.

## Task 3. What goes with the price?

The data are clean. Now we look for the features that go together with the label, the price: these are the features a model would need. Often we draw a plot on the data *before* cleaning, and then on `clean`: that shows what the cleaning was good for.

### The distribution of the price

**Your turn.** Draw a histogram of the clean prices. `bins` is the number of bars, a whole number; choose it yourself: Lecture 2 suggested to try a few values, such as 5 and 100. What changes? Then label both axes, with units.

In [ ]:
fig, ax = plt.subplots()
ax.hist(clean["price"], bins=...)
ax.set_xlabel(...)
ax.set_ylabel(...)
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

- `bins=50` draws 50 bars. Too few bars hide the shape; too many make it noisy.
- Labels are text in quotes, for example `ax.set_xlabel("price per night (euros)")`.

</details>

In [ ]:
# Check
check_3a()

**Think.** Describe the shape of the distribution: where are most prices, and what happens on the right? Will the mean price be above or below the median price, and which of the two would you tell your friend?

*Your answer:*

### A map of the city

Longitude on the x-axis, latitude on the y-axis, one dot per listing, and the price as color.

**Run.** First before cleaning: all listings with a price, outliers included.

In [ ]:
before = listings[listings["price"].notna()]   # numbers, but not cleaned

fig, ax = plt.subplots(figsize=(7, 5))
# c: the values that set the colors; cmap: the colormap; s: the size of the dots (a number)
points = ax.scatter(before["longitude"], before["latitude"], c=before["price"], cmap="viridis", s=3)
fig.colorbar(points, ax=ax, label="price per night (euros)")   # the color scale, with its label
ax.set_aspect(1.6)      # one degree of longitude is shorter than one of latitude here
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("Before cleaning")
plt.show()

Almost every dot has the same dark color: the colorbar runs to almost €12,000, so a single listing decides the colors of all the others. After cleaning, the most expensive listing costs just under €2,000, which still squeezes most colors together.

**How colors work in a scatter plot.** A few arguments decide the colors:

| Argument | What it does |
|---|---|
| `c=...` | the values that decide the color of each dot, here the prices |
| `cmap="..."` | the *colormap*: which colors to use, from the lowest value to the highest. For example `"viridis"` runs from dark purple to yellow, and `"RdBu"` from red through white to blue. |
| `vmin=...`, `vmax=...` | the values at the bottom and the top of the color scale. Values below `vmin` get the bottom color, values above `vmax` the top color. Without them, matplotlib uses the smallest and the largest value, so one extreme listing can decide all the colors, as in the map above. |
| `_r` at the end of a name | reverses a colormap: `"RdBu"` runs from red (low) to blue (high), `"RdBu_r"` from blue (low) to red (high). For prices we use `"RdBu_r"`, so that red means *expensive*, as readers expect. |

`fig.colorbar(...)` draws the color scale next to the plot, so that a reader can translate the colors back into euros.

Lecture 2 showed three kinds of colormaps, each for a different kind of data:

- **sequential** (such as `viridis`): amounts, from little to much;
- **diverging** (such as `RdBu`): differences above and below a midpoint;
- **qualitative** (the default colors of matplotlib and seaborn): groups without an order.

And avoid rainbow colormaps such as `jet`: their sharp color edges suggest boundaries that are not in the data (Lecture 2, "Avoid the rainbow").

**Run.** Amsterdam has seven districts (*stadsdelen*), each made up of a few of the 22 neighborhoods. This cell adds the district of each listing as a new column, for the third map below.

In [ ]:
district = {
    "Centrum-West": "Centrum", "Centrum-Oost": "Centrum",
    "Westerpark": "West", "Bos en Lommer": "West", "De Baarsjes - Oud-West": "West",
    "Geuzenveld - Slotermeer": "Nieuw-West", "Osdorp": "Nieuw-West",
    "De Aker - Nieuw Sloten": "Nieuw-West", "Slotervaart": "Nieuw-West",
    "Zuid": "Zuid", "De Pijp - Rivierenbuurt": "Zuid", "Buitenveldert - Zuidas": "Zuid",
    "Oud-Oost": "Oost", "Watergraafsmeer": "Oost",
    "Oostelijk Havengebied - Indische Buurt": "Oost", "IJburg - Zeeburgereiland": "Oost",
    "Noord-West": "Noord", "Oud-Noord": "Noord", "Noord-Oost": "Noord",
    "Bijlmer-Centrum": "Zuidoost", "Bijlmer-Oost": "Zuidoost", "Gaasperdam - Driemond": "Zuidoost",
}
clean = clean.assign(district=clean["neighbourhood"].map(district))   # a new column
clean["district"].value_counts()

**Your turn.** Three maps of the clean data, side by side, one for each kind of colormap. Replace the dots:

1. **The price:** a sequential colormap, with a `vmax` of your choice. Try a few values: which one shows the most?
2. **Above or below the median price:** `diff` is each price minus the median. Choose a diverging colormap, with `vmin` and `vmax` equally far from 0, so that the median price gets the middle color. (With `"RdBu_r"`, red means expensive.)
3. **The district:** seaborn colors groups with `hue=...` and a qualitative palette. Nothing to fill in.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
x, y = clean["longitude"], clean["latitude"]

# 1. price: sequential
points = axes[0].scatter(x, y, c=clean["price"], cmap=..., vmax=..., s=3)
fig.colorbar(points, ax=axes[0], label="price per night (euros)")

# 2. above or below the median price: diverging
diff = clean["price"] - clean["price"].median()
points = axes[1].scatter(x, y, c=diff, cmap=..., vmin=..., vmax=..., s=3)
fig.colorbar(points, ax=axes[1], label="euros above (+) or below (-) the median")

# 3. district: qualitative (hue: one color per district; linewidth=0: no edge around the dots)
sns.scatterplot(data=clean, x="longitude", y="latitude", hue="district", s=4,
                linewidth=0, ax=axes[2])
sns.move_legend(axes[2], "upper left", bbox_to_anchor=(1, 1), markerscale=3)   # legend to the right, bigger dots

for ax, title in zip(axes, ["Price", "Above or below the median price", "District"]):
    ax.set_title(title)
    ax.set_aspect(1.6)
    ax.set_xticks([])     # no numbers on the axes: degrees mean little to a reader
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

- Colormap names go in quotes: `cmap="viridis"`, `cmap="RdBu_r"`.
- For `vmax`, try a few hundred euros, for example `vmax=600`.
- For map 2, `vmin=-300, vmax=300` puts 0 in the middle.

</details>

In [ ]:
# Check
check_3b()

Most tourists would guess that Centrum is the most expensive district. A map shows *where*, but it is hard to compare districts on it. For that, Lecture 2 used a bar chart: one number per group.

**Your turn.** Draw the median price per district as sorted horizontal bars. Follow the slide "Bar chart: one number per group" of Lecture 2: there, the groups were the study programs and the number was the mean grade. Here the groups are the districts (column `district`), and the number is the median price (why the median? Task 2, step 4).

In [ ]:
medians = ...     # one number per district: the median price, sorted

fig, ax = plt.subplots()
...               # horizontal bars
ax.set_xlabel("median price per night (euros)")
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

The slide of Lecture 2:

```python
means = df.groupby("program")["grade"].mean().sort_values()
ax.barh(means.index, means.values)
```

`groupby("program")` splits the table into one group per program, `["grade"]` takes the grades, `.mean()` averages them per group, and `.sort_values()` sorts the result. Change the table, the two column names, and `.mean()` into `.median()`.

</details>

In [ ]:
# Check
check_3c()

**Think.** Use your bar chart and map 3 to read maps 1 and 2: in which districts are the expensive listings, and in which the cheap ones? Is Centrum the most expensive district? Can you think of a reason?

*Your answer:*

### Many features: which ones matter?

The table has eleven numeric features besides the price, and you could draw a scatter plot of each against the price. But real datasets can have thousands of features: in biology, a dataset of gene activity has one column per gene, about 20,000 of them. No one looks at 20,000 scatter plots. A common first step is to *rank* the features by their correlation with the target, and to look closely at the top of the list.

Recall from Lecture 2: a correlation lies between −1 and 1. It is positive when two columns tend to rise together, negative when one tends to fall as the other rises, and close to 0 when there is no straight-line relation.

**Run.** The correlation of each feature with the price, before and after cleaning.

In [ ]:
features = ["accommodates", "bedrooms", "beds", "minimum_nights", "maximum_nights",
            "availability_365", "number_of_reviews", "review_scores_rating",
            "host_listings", "latitude", "longitude"]
ranking = pd.DataFrame({
    "before cleaning": before[features].corrwith(before["price"]),
    "after cleaning": clean[features].corrwith(clean["price"]),
}).sort_values("after cleaning")

fig, ax = plt.subplots(figsize=(7, 5))
ranking.plot.barh(ax=ax)                       # one pair of bars per feature
ax.axvline(0, color="black", linewidth=0.8)    # a vertical line at 0
ax.set_xlabel("correlation with the price")
plt.show()

After cleaning, the size of the home (`bedrooms`, `accommodates`, `beds`) is clearly at the top, and most correlations became stronger. Ten rows and two codes were enough to hide part of the relations: `maximum_nights`, for example, only shows a correlation once the code 2,147,483,647 is gone.

The ranking only shows each feature against the price. To see how the features relate to *each other*, Lecture 2 used a heatmap of the correlation table: one colored cell for every pair of columns.

**Your turn.** Draw the correlation table of a few columns as a heatmap. Correlations run from −1 to 1, with a natural midpoint at 0: choose the kind of colormap that fits, and set `vmin` and `vmax` so that 0 gets the middle color (Lecture 2, "Heatmap: a table of numbers"). Here there is no "expensive" side, so the plain `"RdBu"` is fine: blue for positive correlations, red for negative ones, as in Lecture 2. The colorbar tells the reader which is which.

In [ ]:
cols = ["price", "accommodates", "bedrooms", "beds", "review_scores_rating", "number_of_reviews"]
corr = clean[cols].corr()          # the correlation table: one number per pair of columns

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap=..., vmin=..., vmax=...)   # draws the table as colored cells
fig.colorbar(im, ax=ax, label="correlation")
ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right")   # column names, turned 45 degrees
ax.set_yticks(range(len(cols)), cols)
for i in range(len(cols)):          # write the numbers in the cells
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

A diverging colormap, such as `"RdBu"`, with `vmin=-1` and `vmax=1`: then 0 is white, positive correlations are blue, and negative ones red.

</details>

In [ ]:
# Check
check_3d()

**Think.** Two things to consider. (a) Look at the three columns about the size of the home: `accommodates`, `bedrooms`, `beds`. How strongly do they go together? Does a model need all three? (b) In the ranking, `latitude` and `longitude` are close to 0. Does that mean that location does not matter? Look back at your maps.

*Your answer:*

### A feature against the label

Lecture 2 called a scatter plot of a feature against the label "the most important plot of supervised learning": one dot per example, the feature on the x-axis, the label on the y-axis.

**Your turn.** Choose one feature from the ranking above, for example `"bedrooms"`, `"review_scores_rating"` or `"number_of_reviews"`, and draw it against the price, as on the slide "Scatter plot: two numbers". With thousands of dots, many lie on top of each other: make them transparent with `alpha` (from 0, invisible, to 1, solid).

In [ ]:
feature = ...     # a column name in quotes

fig, ax = plt.subplots()
...               # the scatter plot: the feature on the x-axis, the price on the y-axis
ax.set_xlabel(feature)
ax.set_ylabel("price per night (euros)")
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

The slide of Lecture 2:

```python
ax.scatter(df["study_hours"], df["grade"], alpha=0.6)
```

Here the table is `clean`, the x-values are `clean[feature]` and the y-values `clean["price"]`. Try a few values of `alpha`, such as 0.6 and 0.1: which one shows where most listings are?

</details>

In [ ]:
# Check
check_3e()

**Think.** What do you see? Does the plot agree with the correlation of your feature in the ranking? Then try a second feature.

*Your answer:*

### A first regression line

Here is the same kind of plot for the number of guests against the price, now with a *regression line*: the straight line that best summarizes how the price changes with the feature. Its *slope* says how many euros more per extra guest. This line is already a first model: give it a number of guests, and it predicts a price. Lecture 4 shows how such a line is found. Today we only draw it, before and after cleaning. `np.polyfit(x, y, 1)` finds the line (a polynomial of degree 1), and gives its slope and intercept.

Lecture 2 named two reasons to make a plot: to *explore* the data yourself, and to *present* a finding to someone else. So far you explored. This plot you will show to your friend, so it must be readable without you next to it (Lecture 2, "From default to readable").

**Your turn.** The plot is drawn for you. Add the labels of both axes, with units, and a title that states what the plot shows, not just its topic.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(before["accommodates"], before["price"], alpha=0.1, s=10)   # alpha: from 0 (invisible) to 1 (solid)
guests = np.array([1, 16])          # draw the lines from 1 to 16 guests
for data, color, name in [(before, "red", "before cleaning"), (clean, "black", "after cleaning")]:
    slope, intercept = np.polyfit(data["accommodates"], data["price"], 1)   # 1: a straight line
    ax.plot(guests, intercept + slope * guests, color=color,
            label=f"{name}: {slope:.0f} euros per extra guest")   # the text in the legend
ax.set_ylim(0, 2000)      # (bottom, top) of the y-axis: zoom in on prices up to 2,000
ax.legend()               # shows the labels of the lines

ax.set_xlabel(...)
ax.set_ylabel(...)
ax.set_title(...)
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

A title that states the finding reads like a short sentence, for example "Larger homes cost more per night", rather than "Price and guests".

</details>

In [ ]:
# Check
check_3f()

**Think.** We left out 10 listings of more than 6,000, and the slope changed by about €16 per guest. Why can so few points move the line so much? For which homes does it matter most?

*Your answer:*

## Task 4. Reviews over time

Guests write a review after their stay, so the number of reviews tells you roughly how many guests Airbnb had in Amsterdam.

**Run.** Load the reviews, one row per review, and count them per month: `resample("MS")` groups the dates by month (`MS`: month start), just like `groupby` groups by a column, and `size()` counts the rows in each group.

In [ ]:
reviews = load_reviews()                                   # one row per review: listing and date
monthly = reviews.set_index("date").resample("MS").size()   # number of reviews per month
monthly.tail()

**Your turn.** The line is drawn for you (Lecture 2, "Line plot: change over time": a line, because the months have an order). Fill in the labels of both axes.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly.values)
ax.set_xlabel(...)
ax.set_ylabel(...)
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

What is on each axis? The months on the x-axis, and the number of reviews in each month on the y-axis. For example `ax.set_ylabel("reviews per month")`.

</details>

In [ ]:
# Check
check_4()

**Think.** (a) Describe three things you see. Is each one something real about Amsterdam, or something about how the data were collected? Look closely at the last point. (b) Lecture 1, "Generalization": suppose a model had learned prices from the listings of 2020. Would you trust it for your friend today?

*Your answer:*

### Two series, two axes

A listing's first review roughly marks when it started on Airbnb. So counting first reviews per month gives the number of new listings per month (of the listings that are still online today).

**Run.**

In [ ]:
first = pd.to_datetime(listings["first_review"])                 # text -> dates
new_listings = pd.Series(1, index=first.dropna()).resample("MS").sum()   # count first reviews per month

m = monthly["2022-01":"2026-08"]          # only complete months, from 2022 on
n = new_listings["2022-01":"2026-08"]
print("reviews per month:      ", m.min(), "to", m.max())
print("new listings per month: ", n.min(), "to", n.max())

The two series have very different sizes: thousands of reviews, but tens or hundreds of new listings per month. On one y-axis, the new listings would be a flat line at the bottom. Lecture 2 showed the way out, and its danger: *twin axes*, a second y-axis on the right with its own scale (Lecture 2, "Twin axes: use with care").

**Your turn.** The plot is drawn for you. With two y-axes, the reader must see which line belongs to which axis. Fill in the label of each axis, and give the second line and its label the same color.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(m.index, m.values, color="tab:blue")
ax1.set_ylabel(..., color="tab:blue")

ax2 = ax1.twinx()                         # a second y-axis, on the right
ax2.plot(n.index, n.values, color=...)
ax2.set_ylabel(..., color=...)
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

Labels and colors are text in quotes, for example `ax2.set_ylabel("new listings per month", color="tab:red")`, and the same `color="tab:red"` in `ax2.plot(...)`.

</details>

In [ ]:
# Check
check_4b()

**Run.** The same two series again. The only difference is one line at the end: the right axis now runs from 0 to 1,500.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(m.index, m.values, color="tab:blue")
ax1.set_ylabel("reviews per month", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(n.index, n.values, color="tab:red")
ax2.set_ylabel("new listings per month", color="tab:red")
ax2.set_ylim(0, 1500)     # (bottom, top) of the right axis
plt.show()

**Think.** Same data, two stories. In your plot the lines move together; here, the red line looks flat and unrelated. Who chose the story? And in the data the two series do rise and fall together every year (their correlation is about 0.8). Do reviews create new listings? What drives both?

*Your answer:*

## Task 5. Back to your friend

Time for a first answer to your friend's question. A simple way: look at listings that are *similar* to her apartment, and take their median price.

**Run.** The neighborhoods, so that you can pick one.

In [ ]:
sorted(clean["neighbourhood"].unique())

**Your turn.** Fill in the neighborhood of her apartment (choose one from the list, in quotes, exactly as written) and the number of guests. The code finds the similar listings, whole homes in the same neighborhood for the same number of guests, and plots their prices.

In [ ]:
neighbourhood = ...      # for example "De Pijp - Rivierenbuurt"
guests = ...             # for example 2

similar = clean[(clean["neighbourhood"] == neighbourhood)
                & (clean["room_type"] == "Entire home/apt")
                & (clean["accommodates"] == guests)]
print(len(similar), "similar listings, median price:", similar["price"].median())

fig, ax = plt.subplots()
ax.hist(similar["price"], bins=20)
ax.axvline(similar["price"].median(), color="red", label="median")   # a vertical line at the median
ax.set_xlabel("price per night (euros)")
ax.set_ylabel("similar listings")
ax.legend()
plt.show()

**Think.** What price would you suggest, and how sure are you? Look at the number of similar listings and at the spread of their prices. Then try a neighborhood far from the center, such as `"Bijlmer-Centrum"`: what changes? Lecture 1 asked: should you use machine learning at all? When is this simple answer good enough, and when would you want a model?

*Your answer:*

## Check your understanding

Every lecture ends with a few exam-style questions, and so does this lab. Answer each question for yourself first, then unfold the answer. We go through them together at the end of the lab.

**1.** Which chart fits each question best: a line plot, a scatter plot, a histogram, a bar chart, or box plots side by side? (As in Exercise 3.1 of Lecture 2.)

(a) How did the number of Airbnb guests in Amsterdam change since 2010?<br>
(b) Do listings for more guests cost more?<br>
(c) Which neighborhood has the highest median price?<br>
(d) How are the ratings distributed?<br>
(e) How do the prices of the four room types compare, including the listings that stand out?

<details><summary><b>Answer</b> (click to unfold)</summary>

(a) Line plot: change over time, as your reviews per month in Task 4. (b) Scatter plot: the relation between two numbers, as your feature against the price in Task 3. (c) Bar chart, sorted: one number per group, as your median price per district in Task 3. (d) Histogram: the distribution of one number, as your prices in Task 3. (e) Box plots side by side: several distributions at once, with the outliers as dots, as in Task 2, step 4. This is Exercise 3.1 of Lecture 2, and today you drew every one of these charts yourself.

</details>

**2.** A table of sensor readings has a column `temperature` in °C. `describe()` shows a mean of 412 and a maximum of 9999. What is the most likely explanation, and what do you do first?

A. A heat wave: keep the values &nbsp; B. 9999 is a code for a missing reading: replace it with `np.nan` &nbsp; C. Some readings are in Fahrenheit &nbsp; D. Duplicate rows

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** 9999 is a typical "no value" code, like 2,147,483,647 in `maximum_nights` (Task 2, step 3). Every computation treats it as a temperature: here it pushes the mean to 412. Replace it with `np.nan` before you compute or plot anything.

</details>

**3.** One in a hundred bank transactions is fraud. A model labels *every* transaction as "not fraud". Which share of the transactions does it label correctly?

A. 1% &nbsp; B. 50% &nbsp; C. 99% &nbsp; D. It cannot be computed

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** 99 of every 100 transactions are not fraud, so the model is right 99% of the time, and yet it finds no fraud at all. With rare cases, a high share of correct answers says little. And the rare cases are the outliers that matter: removing "strange" transactions before training would remove exactly what the model must find.

</details>

**4.** Over the months of the year, the sales of ice cream and the number of people with sunburn are strongly correlated. What is the most likely explanation?

A. Ice cream causes sunburn &nbsp; B. Sunburn makes people buy ice cream &nbsp; C. A third variable, the summer, drives both &nbsp; D. Pure coincidence

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** Sunny months bring both ice cream and sunburn: a hidden (latent) variable behind the correlation, like the room type behind the district prices in Task 3. Correlation is not causation (Lecture 2).

</details>

**5.** Which kind of colormap fits each map: sequential, diverging or qualitative?

(a) The number of listings per neighborhood.<br>
(b) The change in the number of listings since last year, which can be negative or positive.<br>
(c) The district each listing belongs to.

<details><summary><b>Answer</b> (click to unfold)</summary>

(a) Sequential: amounts, from few to many. (b) Diverging: deviations from a midpoint (no change), with `vmin` and `vmax` equally far from 0. (c) Qualitative: groups without an order. (Lecture 2, "Three kinds of colormaps"; Task 3.)

</details>

**6.** Supervised or unsupervised? If supervised: classification or regression? (Lecture 1)

(a) Predict the price per night of a new listing.<br>
(b) Predict whether a host will become a superhost.<br>
(c) Group listings with similar descriptions, without any labels.<br>
(d) Predict the number of reviews in Amsterdam next month.

<details><summary><b>Answer</b> (click to unfold)</summary>

(a) Supervised, regression: the answer is an amount. (b) Supervised, classification: the answer is a kind (yes or no). (c) Unsupervised: clustering, there is no label. (d) Supervised, regression: a forecast of an amount over time.

</details>

## If you finish early

Here are two more tasks. You can also do them at home and compare your work with the solutions on Canvas.

**A. Your own question.** Ask the clean data **one** question, and answer it with **one** plot. A reader should understand the plot without your code (Lecture 2, "From default to readable"): label both axes with units, and give the plot a title that states your finding. Some ideas: How are the ratings distributed? Are listings with more reviews rated higher? Which neighborhoods are cheapest *per guest*?

In [ ]:
# My question: ...

fig, ax = plt.subplots()
...
plt.show()

In [ ]:
# Check
check_own()

**B. Superhosts.** Superhosts are experienced, highly rated hosts. Run the cell: do superhosts ask more or less than other hosts? Then think of a hidden variable, as in Task 3, and test your idea.

In [ ]:
clean.groupby("host_is_superhost")["price"].median()     # t: superhost, f: other hosts

In [ ]:
# test your idea
...

<details><summary><b>Hint</b> (click to unfold)</summary>

`clean.groupby(["room_type", "host_is_superhost"])["price"].median()` gives the median for every combination of room type and superhost.

</details>

## Where this is going

> Plot the data before modeling it. Plots reveal problems that summary numbers hide, and a few lines of `matplotlib` are enough.

Today you took the first steps of the machine learning workflow of Lecture 1: you turned a question into a learning problem (label: the price), looked at the data, and prepared them. You found text that should be numbers, missing values with a reason, a code that is not a measurement, and outliers that pull a line. You cleaned them step by step, and you saw the difference in the plots. No model repairs such problems for you: garbage in, garbage out.

**Coming up.** Before the next lab there are two lectures:

- Lecture 3 (Monday, November 2): supervised learning, your first model ($k$-nearest neighbors: predict from the most similar examples, as you did by hand in Task 5), and how to test a model on new data.
- Lecture 4 (Tuesday, November 3): linear models for regression: how the line of Task 3 is found.

Laptop lab 2 (Thursday, November 5) practices both lectures.

**Reading.** Müller and Guido (2016), Chapter 1: "Essential Libraries and Tools" and "First Things First: Look at Your Data".

---
*This lab practices Lectures 1 and 2 of Ş. İlker Birbil. Data: [Inside Airbnb](https://insideairbnb.com), Amsterdam, collected on September 16, 2026, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); downloaded directly from their website, and we keep 20 of the 90 columns of the listings.*